Python script that processes other datasets for annotation. Based on [dataset_processing](dataset_processing.py) and [parse_ccner_datasets](parse_ccner_datasets.ipynb). Outputs are stored in RESULTS.

In [1]:
from dataset_processing import BIODIVNER_DIR, BIODIVNER_LABELS, biodivner_process_bio_documents
from dataset_processing import IBMCCNER_DIR, IBMCCNER_LABELS, ibmccner_process_bio_documents 
from dataset_processing import convert_to_labelstudio_format

from datasets import load_dataset

import json

import os
import spacy

## Loading BioDivNER

Using a modified `load_biodivner()` function from `dataset_processing`:

In [ ]:
split = ["train", "test", "dev"]
print(f"Loading '{BIODIVNER_DIR}' with splits: {split}")

if type("string") == type(split):
    split = [split]
    
print("Converting from BIO tags to char spans.")
biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=BIODIVNER_DIR, 
    labels_to_keep=BIODIVNER_LABELS, 
    split=split
    )

for i in range(len(biodivner_structured_data)):
    biodivner_structured_data[i]["sentence"] = biodivner_structured_data[i]["text"]
    biodivner_structured_data[i]["sentence_id"] = i
    
ls_biodivner = convert_to_labelstudio_format(biodivner_structured_data, "0")

with open("RESULTS/BioDivNER/biodivner_full.json", "w", encoding="utf-8") as outfile:
    json.dump(ls_biodivner, outfile, indent=4)

## Loading IBMCCNER


Using a modified `load_ibmccner()` function from `dataset_processing`:

In [ ]:
# 1. Load and pre-process the data into a document-wise list of lines
split = ["train", "validation", "test"]
print(f"Loading '{IBMCCNER_DIR}' with splits: {split}")
ds = load_dataset(IBMCCNER_DIR) 
train_documentwise = []
temp_list = []
if type(split) == type(["list"]):
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith('-DOCSTART-'):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
else:
    for line in ds[split]["text"]:
        if line.strip().startswith('-DOCSTART-'):
            if temp_list:
                train_documentwise.append(temp_list)
            temp_list = []
        temp_list.append(line)
    if temp_list:
        train_documentwise.append(temp_list)

print("Converting from BIO tags to char spans.")        
structured_documents_char_spans = ibmccner_process_bio_documents(
    document_list=train_documentwise,
    labels_to_keep=IBMCCNER_LABELS
)

for i in range(len(structured_documents_char_spans)):
    structured_documents_char_spans[i]["sentence"] = structured_documents_char_spans[i]["text"]
    structured_documents_char_spans[i]["sentence_id"] = i

ls_ibmccner = convert_to_labelstudio_format(structured_documents_char_spans, "0")

with open("RESULTS/IBMCCNER/ibmccner_full.json", "w", encoding="utf-8") as outfile:
    json.dump(ls_ibmccner, outfile, indent=4)

Loading 'ibm-research/Climate-Change-NER' with splits: ['train', 'validation', 'test']
Converting from BIO tags to char spans.


## Load ClimateIE

In [2]:
# Constants
DATA_DIRECTORY = "DATA/ClimateIE/human_corpus/"

# 1. Setup SpaCy
print("Importing SpaCy ...")
# Using sci_sm as requested, disabling ner for speed since we have our own labels
nlp = spacy.load("en_core_sci_sm", disable=["ner"])

try:
    file_names = [f for f in os.listdir(DATA_DIRECTORY) if f.endswith('.json')]
except FileNotFoundError:
    print(f"Error: The directory '{DATA_DIRECTORY}' was not found.")
    file_names = []

# This list will hold the final transformed data for all files
structured_documents_char_spans = []

# Process each file
for file_name in file_names:
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    
    with open(file_path, "r", encoding="utf-8") as f:
        document = json.load(f)
    
    # Raw text and global entities dictionary
    raw_text = document["text"]
    raw_entities = document["entities"] # This is a dict of ID -> {begin, end, ...}

    # 2. Tokenize into sentences
    # Increasing max_length in case documents are very large
    nlp.max_length = len(raw_text) + 100000 
    doc = nlp(raw_text)

    # Convert the entities dict to a list for easier iteration
    # We store the ID inside the dict for reference
    entity_list = []
    for ent_id, ent_data in raw_entities.items():
        ent_data['id_key'] = ent_id # Keep the original key just in case
        entity_list.append(ent_data)

    # Sort entities by start position to potentially optimize or ensure order
    entity_list.sort(key=lambda x: x['begin'])

    sentence_counter = 0

    # Iterate over sentences found by SpaCy
    for sent in doc.sents:
        sent_text = sent.text
        # These are the global offsets of the sentence within the document
        sent_start_char = sent.start_char
        sent_end_char = sent.end_char
        
        valid_local_entities = []

        # 3. Remap entities to sentences
        for ent in entity_list:
            # Check if the entity is strictly contained within this sentence
            if ent['begin'] >= sent_start_char and ent['end'] <= sent_end_char:
                
                # Calculate new offsets relative to the sentence
                local_start = ent['begin'] - sent_start_char
                local_end = ent['end'] - sent_start_char
                
                # 4. Transform data format (including Identifier)
                valid_local_entities.append({
                    "text": ent['substring'],
                    "label": ent['label'],
                    "start": local_start,
                    "end": local_end,
                    "identifier": ent.get('identifier', None) # Bonus: Keep identifier
                })

        # Only add sentences that have text (skip empty newlines)
        if sent_text.strip():
            structured_documents_char_spans.append({
                "text": sent_text,
                "sentence": sent_text, # Redundant but matches your requested format
                "sentence_id": sentence_counter,
                "entities": valid_local_entities
            })
            sentence_counter += 1

# Output verification
print(f"Processing complete. Total sentences collected: {len(structured_documents_char_spans)}")



Importing SpaCy ...


/home/p0l3/miniforge3/envs/clirener_finetune/lib/python3.10/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Processing complete. Total sentences collected: 6552


In [3]:
# 2. systematic Sanity Check Loop
print("\n" + "-" * 30)
print("Starting Global Sanity Check...")

errors_found = 0

for idx, item in enumerate(structured_documents_char_spans):
    # The full sentence text
    sentence_text = item["text"]
    
    # Iterate over every entity in this sentence
    for entity in item["entities"]:
        start = entity["start"]
        end = entity["end"]
        expected_text = entity["text"]
        
        # SLICE the sentence using the indices
        actual_slice = sentence_text[start:end]
        
        # COMPARE
        if actual_slice != expected_text:
            errors_found += 1
            print(f"Mismatch found at Index {idx}!")
            print(f"  Sentence: {sentence_text}")
            print(f"  Indices:  [{start}:{end}]")
            print(f"  Expected: '{expected_text}'")
            print(f"  Actual:   '{actual_slice}'")

if errors_found == 0:
    print("Sanity Check PASSED: All entity slices match their text labels exactly.")
else:
    print(f"Sanity Check FAILED: {errors_found} mismatches found.")


------------------------------
Starting Global Sanity Check...
Sanity Check PASSED: All entity slices match their text labels exactly.


In [4]:
ls_climateie = convert_to_labelstudio_format(structured_documents_char_spans, "0")

with open("RESULTS/CLIMATEIE/climateie_full.json", "w", encoding="utf-8") as outfile:
    json.dump(ls_climateie, outfile, indent=4)